### DATASET 5: Remove ANA, AntidsDNA and AntiSm, Renal biopsy, APL, C3, C4 IN EULAR/ACR 2019 criteria IN EULAR/ACR 2019 criteria

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#Load dataset
df = pd.read_csv('SLE_NotSLE.csv')
df2 = pd.read_csv('SLE_NotSLE.csv')

In [3]:
#Get Columns
Columns = df.columns
print(Columns)

Index(['Sex', 'Age', 'Fever', 'ACL', 'SCL or DL', 'Oral Ulcer', 'Alopecia',
       'Joint involvement', 'Acute pericarditis',
       'Pleural or pericardial effusion', 'Proteinuria',
       'Renal class II or V LN', 'Renal class III or IV LN', 'Delirium',
       'Psychosis', 'Seizure', 'Leukopenia', 'Thrombocytopenia', 'AIHA',
       'Antiphospholipid', 'Low C3 or C4', 'Low C4 and C3',
       'Anti-dsDNA or Anti-Sm', 'ANA', 'Diagnosis'],
      dtype='object')


In [4]:
display(df.head())
display(df.tail())

,Sex,Age,Fever,ACL,SCL or DL,Oral Ulcer,Alopecia,Joint involvement,Acute pericarditis,Pleural or pericardial effusion,...,Seizure,Leukopenia,Thrombocytopenia,AIHA,Antiphospholipid,Low C3 or C4,Low C4 and C3,Anti-dsDNA or Anti-Sm,ANA,Diagnosis
0,0,31,1,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,1,1,1
1,0,27,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,1,1,1
2,0,43,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,1,1,1,1
3,0,27,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,1,1,1
4,0,36,0,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,1,1,1


,Sex,Age,Fever,ACL,SCL or DL,Oral Ulcer,Alopecia,Joint involvement,Acute pericarditis,Pleural or pericardial effusion,...,Seizure,Leukopenia,Thrombocytopenia,AIHA,Antiphospholipid,Low C3 or C4,Low C4 and C3,Anti-dsDNA or Anti-Sm,ANA,Diagnosis
397,0,37,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
398,0,46,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
399,0,59,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
400,0,60,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
401,0,63,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
#Remove Age and Sex
#Remove column
df = df.drop(columns=['Sex', 'Age','Anti-dsDNA or Anti-Sm', 'Antiphospholipid', 'Low C3 or C4', 'Low C4 and C3', 'ANA', 'Renal class II or V LN', 'Renal class III or IV LN',])
df.columns

Index(['Fever', 'ACL', 'SCL or DL', 'Oral Ulcer', 'Alopecia',
       'Joint involvement', 'Acute pericarditis',
       'Pleural or pericardial effusion', 'Proteinuria', 'Delirium',
       'Psychosis', 'Seizure', 'Leukopenia', 'Thrombocytopenia', 'AIHA',
       'Diagnosis'],
      dtype='object')

In [6]:
#Determining X and Y variables
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [7]:
#Import Classifier
from sklearn.tree import DecisionTreeClassifier
treeClf = DecisionTreeClassifier()

from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier(bootstrap= True, max_depth = 10, min_samples_leaf = 1, min_samples_split = 2, n_estimators = 300)

from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier()

from sklearn.svm import SVC
svmClf = SVC(C= 10, gamma = 1, kernel = 'rbf', probability=True)

In [8]:
#Import neccessary libraries for training and evaluation
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix


In [9]:
#Bootstrapped Confidence Intervals function

def bootstrap_ci(data, n_bootstrap=1000, ci=0.95):
    boot_means = []
    n = len(data)
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        boot_means.append(np.mean(sample))
    lower = np.percentile(boot_means, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return np.mean(data), lower, upper

In [10]:
#Set K-Fold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [11]:
#Train and Test Standardized criteria

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, roc_auc_score, confusion_matrix
)

# -------------------------
# 1. Define SLE classification function
# -------------------------
def sle_classification_2019_no_entry(data):
    """
    Classify SLE per 2019 EULAR/ACR criteria (without ANA entry criterion).
    Returns (raw_score, binary_label).
    """

    domains ={
        'Constitutional': {'Fever': 2}, 
        'Mucocutaneous': {'ACL': 6,'SCL or DL': 4, 'Oral Ulcer': 2,'Alopecia': 2},
        'Musculoskeletal': {'Joint involvement': 6},
        'Serosal': {'Acute pericarditis': 6, 'Pleural or pericardial effusion': 5},
        'Renal': {'Proteinuria': 4},
        'Neuropsychiatric': {'Delirium': 2, 'Psychosis': 3, 'Seizure': 5},
        'Hematologic': {'Leukopenia': 3, 'Thrombocytopenia': 4, 'AIHA': 4},
        
    }

    total = 0
    for domain, items in domains.items():
        present_scores = [pts for key, pts in items.items() if data.get(key, False)]
        if present_scores:
            total += max(present_scores)

    classified = 1 if total >= 10 else 0
    return total, classified


# -------------------------
# 2. Conversion helper (array → dict)
# -------------------------
FEATURE_ORDER = ['Fever', 'ACL', 'SCL or DL', 'Oral Ulcer', 'Alopecia',
       'Joint involvement', 'Acute pericarditis',
       'Pleural or pericardial effusion', 'Proteinuria',
       'Delirium',
       'Psychosis', 'Seizure', 'Leukopenia', 'Thrombocytopenia', 'AIHA',
       ]

def row_to_dict(row, feature_order):
    """Convert a numpy row (0/1 values) into dict for sle_classification_2019_no_entry."""
    return {feature_order[i]: (row[i] == 1) for i in range(len(feature_order))}


# -------------------------
# 3. Cross-validation evaluation
# -------------------------
kf = KFold(n_splits=10, shuffle=True, random_state=42)

MAX_SCORE = 29  # maximum possible score

accuracies, sensitivities, specificities, precisions, f1_scores, roc_aucs = [], [], [], [], [], []

for train_index, test_index in kf.split(X):
    X_test, y_test = X[test_index], y[test_index]

    y_pred, y_pred_prob = [], []

    for row in X_test:
        data_dict = row_to_dict(row, FEATURE_ORDER)
        score, label = sle_classification_2019_no_entry(data_dict)
        y_pred.append(label)
        y_pred_prob.append(score / MAX_SCORE)  # normalize score to [0,1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # sensitivity
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # Append
    accuracies.append(accuracy)
    sensitivities.append(recall)
    specificities.append(specificity)
    precisions.append(precision)
    f1_scores.append(f1)
    roc_aucs.append(roc_auc)

# -------------------------
# 5. Print mean metrics
# -------------------------
# Calculate mean metrics with 95% CI
mean_accuracy, ci_low_acc, ci_high_acc = bootstrap_ci(accuracies)
mean_sensitivity, ci_low_sens, ci_high_sens = bootstrap_ci(sensitivities)
mean_specificity, ci_low_spec, ci_high_spec = bootstrap_ci(specificities)
mean_precision, ci_low_prec, ci_high_prec = bootstrap_ci(precisions)
mean_f1, ci_low_f1, ci_high_f1 = bootstrap_ci(f1_scores)
mean_auc, ci_low_auc, ci_high_auc = bootstrap_ci(roc_aucs)

# Print results
print(f"Accuracy: {mean_accuracy:.3f} (95% CI: {ci_low_acc:.3f} - {ci_high_acc:.3f})")
print(f"Sensitivity: {mean_sensitivity:.3f} (95% CI: {ci_low_sens:.3f} - {ci_high_sens:.3f})")
print(f"Specificity: {mean_specificity:.3f} (95% CI: {ci_low_spec:.3f} - {ci_high_spec:.3f})")
print(f"Precision: {mean_precision:.3f} (95% CI: {ci_low_prec:.3f} - {ci_high_prec:.3f})")
print(f"F1-Score: {mean_f1:.3f} (95% CI: {ci_low_f1:.3f} - {ci_high_f1:.3f})")
print(f"ROC-AUC: {mean_auc:.3f} (95% CI: {ci_low_auc:.3f} - {ci_high_auc:.3f})")

Accuracy: 0.816 (95% CI: 0.781 - 0.858)
Sensitivity: 0.651 (95% CI: 0.585 - 0.728)
Specificity: 0.980 (95% CI: 0.963 - 0.995)
Precision: 0.968 (95% CI: 0.940 - 0.993)
F1-Score: 0.773 (95% CI: 0.722 - 0.823)
ROC-AUC: 0.828 (95% CI: 0.794 - 0.861)


In [12]:
#Train and Test Decision Tree

# Initialize lists to store metrics
accuracies = []
sensitivities = []  # Sensitivity (Recall)
specificities = []
precisions = []
f1_scores = []
roc_aucs = []

for train_index, test_index in kf.split(X):
    # Split the data into train and test sets
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train the treeClf
    treeClf.fit(X_train, y_train)

    # Make predictions
    y_pred = treeClf.predict(X_test)
    y_pred_prob = treeClf.predict_proba(X_test)[:, 1]  # Get probabilities for ROC-AUC

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # Sensitivity is the same as recall
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Handle zero division

    # Append metrics to lists
    accuracies.append(accuracy)
    sensitivities.append(recall)
    specificities.append(specificity)
    precisions.append(precision)
    f1_scores.append(f1)
    roc_aucs.append(roc_auc)

# Calculate mean metrics across all folds
mean_accuracy, ci_low_acc, ci_high_acc = bootstrap_ci(accuracies)
mean_sensitivity, ci_low_sens, ci_high_sens = bootstrap_ci(sensitivities)
mean_specificity, ci_low_spec, ci_high_spec = bootstrap_ci(specificities)
mean_precision, ci_low_prec, ci_high_prec = bootstrap_ci(precisions)
mean_f1_score, ci_low_f1, ci_high_f1 = bootstrap_ci(f1_scores)
mean_roc_auc, ci_low_auc, ci_high_auc = bootstrap_ci(roc_aucs)


# Print results
print(f"Accuracy: {mean_accuracy:.3f} (95% CI: {ci_low_acc:.3f} - {ci_high_acc:.3f})")
print(f"Sensitivity: {mean_sensitivity:.3f} (95% CI: {ci_low_sens:.3f} - {ci_high_sens:.3f})")
print(f"Specificity: {mean_specificity:.3f} (95% CI: {ci_low_spec:.3f} - {ci_high_spec:.3f})")
print(f"Precision: {mean_precision:.3f} (95% CI: {ci_low_prec:.3f} - {ci_high_prec:.3f})")
print(f"F1-Score: {mean_f1_score:.3f} (95% CI: {ci_low_f1:.3f} - {ci_high_f1:.3f})")
print(f"ROC-AUC: {mean_roc_auc:.3f} (95% CI: {ci_low_auc:.3f} - {ci_high_auc:.3f})")

Accuracy: 0.938 (95% CI: 0.898 - 0.970)
Sensitivity: 0.907 (95% CI: 0.839 - 0.965)
Specificity: 0.965 (95% CI: 0.938 - 0.990)
Precision: 0.963 (95% CI: 0.935 - 0.991)
F1-Score: 0.931 (95% CI: 0.877 - 0.969)
ROC-AUC: 0.955 (95% CI: 0.934 - 0.974)


In [13]:
#Train and Test RandomForestClassifier

# Initialize lists to store metrics
accuracies = []
sensitivities = []  # Sensitivity (Recall)
specificities = []
precisions = []
f1_scores = []
roc_aucs = []

for train_index, test_index in kf.split(X):
    # Split the data into train and test sets
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train the RF
    RF.fit(X_train, y_train)

    # Make predictions
    y_pred = RF.predict(X_test)
    y_pred_prob = RF.predict_proba(X_test)[:, 1]  # Get probabilities for ROC-AUC

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # Sensitivity is the same as recall
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Handle zero division

    # Append metrics to lists
    accuracies.append(accuracy)
    sensitivities.append(recall)
    specificities.append(specificity)
    precisions.append(precision)
    f1_scores.append(f1)
    roc_aucs.append(roc_auc)

# Calculate mean metrics across all folds
mean_accuracy, ci_low_acc, ci_high_acc = bootstrap_ci(accuracies)
mean_sensitivity, ci_low_sens, ci_high_sens = bootstrap_ci(sensitivities)
mean_specificity, ci_low_spec, ci_high_spec = bootstrap_ci(specificities)
mean_precision, ci_low_prec, ci_high_prec = bootstrap_ci(precisions)
mean_f1_score, ci_low_f1, ci_high_f1 = bootstrap_ci(f1_scores)
mean_roc_auc, ci_low_auc, ci_high_auc = bootstrap_ci(roc_aucs)


# Print results
print(f"Accuracy: {mean_accuracy:.3f} (95% CI: {ci_low_acc:.3f} - {ci_high_acc:.3f})")
print(f"Sensitivity: {mean_sensitivity:.3f} (95% CI: {ci_low_sens:.3f} - {ci_high_sens:.3f})")
print(f"Specificity: {mean_specificity:.3f} (95% CI: {ci_low_spec:.3f} - {ci_high_spec:.3f})")
print(f"Precision: {mean_precision:.3f} (95% CI: {ci_low_prec:.3f} - {ci_high_prec:.3f})")
print(f"F1-Score: {mean_f1_score:.3f} (95% CI: {ci_low_f1:.3f} - {ci_high_f1:.3f})")
print(f"ROC-AUC: {mean_roc_auc:.3f} (95% CI: {ci_low_auc:.3f} - {ci_high_auc:.3f})")

Accuracy: 0.953 (95% CI: 0.933 - 0.972)
Sensitivity: 0.941 (95% CI: 0.918 - 0.962)
Specificity: 0.965 (95% CI: 0.936 - 0.992)
Precision: 0.965 (95% CI: 0.937 - 0.990)
F1-Score: 0.952 (95% CI: 0.932 - 0.972)
ROC-AUC: 0.974 (95% CI: 0.961 - 0.989)


In [14]:
#Train and Test KNeighborsClassifier

# Initialize lists to store metrics
accuracies = []
sensitivities = []  # Sensitivity (Recall)
specificities = []
precisions = []
f1_scores = []
roc_aucs = []

for train_index, test_index in kf.split(X):
    # Split the data into train and test sets
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train the neigh
    neigh.fit(X_train, y_train)

    # Make predictions
    y_pred = neigh.predict(X_test)
    y_pred_prob = neigh.predict_proba(X_test)[:, 1]  # Get probabilities for ROC-AUC

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # Sensitivity is the same as recall
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Handle zero division

    # Append metrics to lists
    accuracies.append(accuracy)
    sensitivities.append(recall)
    specificities.append(specificity)
    precisions.append(precision)
    f1_scores.append(f1)
    roc_aucs.append(roc_auc)

# Calculate mean metrics across all folds
mean_accuracy, ci_low_acc, ci_high_acc = bootstrap_ci(accuracies)
mean_sensitivity, ci_low_sens, ci_high_sens = bootstrap_ci(sensitivities)
mean_specificity, ci_low_spec, ci_high_spec = bootstrap_ci(specificities)
mean_precision, ci_low_prec, ci_high_prec = bootstrap_ci(precisions)
mean_f1_score, ci_low_f1, ci_high_f1 = bootstrap_ci(f1_scores)
mean_roc_auc, ci_low_auc, ci_high_auc = bootstrap_ci(roc_aucs)


# Print results
print(f"Accuracy: {mean_accuracy:.3f} (95% CI: {ci_low_acc:.3f} - {ci_high_acc:.3f})")
print(f"Sensitivity: {mean_sensitivity:.3f} (95% CI: {ci_low_sens:.3f} - {ci_high_sens:.3f})")
print(f"Specificity: {mean_specificity:.3f} (95% CI: {ci_low_spec:.3f} - {ci_high_spec:.3f})")
print(f"Precision: {mean_precision:.3f} (95% CI: {ci_low_prec:.3f} - {ci_high_prec:.3f})")
print(f"F1-Score: {mean_f1_score:.3f} (95% CI: {ci_low_f1:.3f} - {ci_high_f1:.3f})")
print(f"ROC-AUC: {mean_roc_auc:.3f} (95% CI: {ci_low_auc:.3f} - {ci_high_auc:.3f})")

Accuracy: 0.565 (95% CI: 0.534 - 0.595)
Sensitivity: 0.944 (95% CI: 0.911 - 0.977)
Specificity: 0.183 (95% CI: 0.127 - 0.235)
Precision: 0.534 (95% CI: 0.495 - 0.571)
F1-Score: 0.680 (95% CI: 0.646 - 0.713)
ROC-AUC: 0.577 (95% CI: 0.500 - 0.662)


In [15]:
#Train and Test SVM

# Initialize lists to store metrics
accuracies = []
sensitivities = []  # Sensitivity (Recall)
specificities = []
precisions = []
f1_scores = []
roc_aucs = []

for train_index, test_index in kf.split(X):
    # Split the data into train and test sets
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train the svmClf
    svmClf.fit(X_train, y_train)

    # Make predictions
    y_pred = svmClf.predict(X_test)
    y_pred_prob = svmClf.predict_proba(X_test)[:, 1]  # Get probabilities for ROC-AUC

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # Sensitivity is the same as recall
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Handle zero division

    # Append metrics to lists
    accuracies.append(accuracy)
    sensitivities.append(recall)
    specificities.append(specificity)
    precisions.append(precision)
    f1_scores.append(f1)
    roc_aucs.append(roc_auc)

# Calculate mean metrics across all folds
mean_accuracy, ci_low_acc, ci_high_acc = bootstrap_ci(accuracies)
mean_sensitivity, ci_low_sens, ci_high_sens = bootstrap_ci(sensitivities)
mean_specificity, ci_low_spec, ci_high_spec = bootstrap_ci(specificities)
mean_precision, ci_low_prec, ci_high_prec = bootstrap_ci(precisions)
mean_f1_score, ci_low_f1, ci_high_f1 = bootstrap_ci(f1_scores)
mean_roc_auc, ci_low_auc, ci_high_auc = bootstrap_ci(roc_aucs)


# Print results
print(f"Accuracy: {mean_accuracy:.3f} (95% CI: {ci_low_acc:.3f} - {ci_high_acc:.3f})")
print(f"Sensitivity: {mean_sensitivity:.3f} (95% CI: {ci_low_sens:.3f} - {ci_high_sens:.3f})")
print(f"Specificity: {mean_specificity:.3f} (95% CI: {ci_low_spec:.3f} - {ci_high_spec:.3f})")
print(f"Precision: {mean_precision:.3f} (95% CI: {ci_low_prec:.3f} - {ci_high_prec:.3f})")
print(f"F1-Score: {mean_f1_score:.3f} (95% CI: {ci_low_f1:.3f} - {ci_high_f1:.3f})")
print(f"ROC-AUC: {mean_roc_auc:.3f} (95% CI: {ci_low_auc:.3f} - {ci_high_auc:.3f})")

Accuracy: 0.953 (95% CI: 0.925 - 0.978)
Sensitivity: 0.939 (95% CI: 0.893 - 0.973)
Specificity: 0.965 (95% CI: 0.938 - 0.990)
Precision: 0.964 (95% CI: 0.936 - 0.991)
F1-Score: 0.950 (95% CI: 0.914 - 0.979)
ROC-AUC: 0.966 (95% CI: 0.949 - 0.981)


In [16]:
import xgboost as xgb

# Initialize lists to store metrics
accuracies = []
sensitivities = []  # Sensitivity (Recall)
specificities = []
precisions = []
f1_scores = []
roc_aucs = []

for train_index, test_index in kf.split(X):
    # Split the data into train and test sets
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    # Define model parameters
    params = {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 7, 'subsample': 0.9}
    
    # Train the model
    bst = xgb.train(params, dtrain, num_boost_round=100)
    
    # Make predictions
    y_pred_prob = bst.predict(dtest)  # Predicted probabilities for ROC-AUC
    y_pred = (y_pred_prob > 0.5).astype(int)  # Convert probabilities to binary predictions
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # Sensitivity is the same as recall
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)  # F1-score
    roc_auc = roc_auc_score(y_test, y_pred_prob)  # ROC-AUC
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Handle zero division
    
    # Append metrics to lists
    accuracies.append(accuracy)
    sensitivities.append(recall)
    specificities.append(specificity)
    precisions.append(precision)
    f1_scores.append(f1)
    roc_aucs.append(roc_auc)

# Calculate average metrics
mean_accuracy, ci_low_acc, ci_high_acc = bootstrap_ci(accuracies)
mean_sensitivity, ci_low_sens, ci_high_sens = bootstrap_ci(sensitivities)
mean_specificity, ci_low_spec, ci_high_spec = bootstrap_ci(specificities)
mean_precision, ci_low_prec, ci_high_prec = bootstrap_ci(precisions)
mean_f1_score, ci_low_f1, ci_high_f1 = bootstrap_ci(f1_scores)
mean_roc_auc, ci_low_auc, ci_high_auc = bootstrap_ci(roc_aucs)


# Print results
print(f"Accuracy: {mean_accuracy:.3f} (95% CI: {ci_low_acc:.3f} - {ci_high_acc:.3f})")
print(f"Sensitivity: {mean_sensitivity:.3f} (95% CI: {ci_low_sens:.3f} - {ci_high_sens:.3f})")
print(f"Specificity: {mean_specificity:.3f} (95% CI: {ci_low_spec:.3f} - {ci_high_spec:.3f})")
print(f"Precision: {mean_precision:.3f} (95% CI: {ci_low_prec:.3f} - {ci_high_prec:.3f})")
print(f"F1-Score: {mean_f1_score:.3f} (95% CI: {ci_low_f1:.3f} - {ci_high_f1:.3f})")
print(f"ROC-AUC: {mean_roc_auc:.3f} (95% CI: {ci_low_auc:.3f} - {ci_high_auc:.3f})")

Accuracy: 0.955 (95% CI: 0.933 - 0.978)
Sensitivity: 0.948 (95% CI: 0.915 - 0.977)
Specificity: 0.965 (95% CI: 0.935 - 0.992)
Precision: 0.966 (95% CI: 0.938 - 0.991)
F1-Score: 0.956 (95% CI: 0.935 - 0.979)
ROC-AUC: 0.972 (95% CI: 0.961 - 0.984)


In [17]:
Columns_last = df.columns
print(Columns_last)
print('number of column =', len(Columns_last))

Index(['Fever', 'ACL', 'SCL or DL', 'Oral Ulcer', 'Alopecia',
       'Joint involvement', 'Acute pericarditis',
       'Pleural or pericardial effusion', 'Proteinuria', 'Delirium',
       'Psychosis', 'Seizure', 'Leukopenia', 'Thrombocytopenia', 'AIHA',
       'Diagnosis'],
      dtype='object')
number of column = 16
